# 예제 04. CPU와 GPU
빅데이터프로그래밍 · 4주차

## 목표
- GPU 사용 가능 여부를 확인한다
- Tensor를 장치 사이로 옮긴다
- 같은 장치에 있어야 계산된다는 규칙을 확인한다
- 속도 차이를 직접 측정한다

**먼저** `런타임 > 런타임 유형 변경 > T4 GPU` 를 선택하세요.


In [ ]:
import torch

print("GPU 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("장치 이름:", torch.cuda.get_device_name(0))
else:
    print("런타임 유형을 T4 GPU 로 바꾸고 다시 실행하세요")


## 1. 장치를 변수로 정해 두는 습관
코드 맨 앞에 한 줄 두면, CPU에서도 GPU에서도 같은 코드가 돌아갑니다.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("사용할 장치:", device)


## 2. Tensor 옮기기


In [ ]:
x = torch.randn(3, 3)
print("만든 직후:", x.device)

x_gpu = x.to(device)
print("옮긴 후  :", x_gpu.device)

# 만들 때 바로 지정할 수도 있습니다
y = torch.randn(3, 3, device=device)
print(y.device)


## 3. 같은 장치에 있어야 계산됩니다
모델과 데이터가 다른 장치에 있으면 학습이 시작되지 않습니다. 6주차에 가장 많이 만나는 오류입니다.


In [ ]:
if torch.cuda.is_available():
    cpu_t = torch.randn(3, 3)
    gpu_t = torch.randn(3, 3, device="cuda")
    try:
        cpu_t + gpu_t
    except RuntimeError as err:
        print("RuntimeError:", err)
else:
    print("GPU 런타임에서 실행하세요")


In [ ]:
# 해결: 한쪽을 옮긴다
if torch.cuda.is_available():
    print((cpu_t.to("cuda") + gpu_t).device)


## 4. GPU 텐서는 NumPy로 바로 못 바꿉니다


In [ ]:
if torch.cuda.is_available():
    t = torch.randn(2, 2, device="cuda")
    try:
        t.numpy()
    except TypeError as err:
        print("TypeError:", err)
    print("해결:", t.cpu().numpy())


## 5. 속도 비교
행렬곱은 GPU가 크게 유리합니다. 크기가 작으면 차이가 없거나 CPU가 빠릅니다.


In [ ]:
import time

def bench(device, n=2000, repeat=3):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(repeat):
        a @ b
    if device == "cuda":
        torch.cuda.synchronize()      # GPU 계산이 끝날 때까지 기다린다
    return (time.time() - start) / repeat

print(f"CPU : {bench('cpu'):.4f} 초")
if torch.cuda.is_available():
    print(f"GPU : {bench('cuda'):.4f} 초")


In [ ]:
# 작은 행렬에서는 차이가 없습니다 — 옮기는 비용이 더 큽니다
print(f"CPU 200x200 : {bench('cpu', n=200):.5f} 초")
if torch.cuda.is_available():
    print(f"GPU 200x200 : {bench('cuda', n=200):.5f} 초")


## 직접 해보기
1. `device` 변수를 써서 (1000, 1000) Tensor를 만들고 device를 출력하세요.
2. 크기를 4000으로 늘려 CPU와 GPU 시간을 비교하세요.


In [ ]:
# 여기에 작성하세요
